# Notebook  — Smart Review Analysis
---

## Functional Code

In [2]:
# -------
# config
# -------
import sys
from pathlib import Path
import torch

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/kaggle/working')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KAGGLE_RAW_CSV  = Path("/kaggle/input/datasets/dongrelaxman/amazon-reviews-dataset/Amazon_Reviews.csv")
KAGGLE_CLEAN_CSV = PROJECT_ROOT / 'data' / 'processed' / 'clean_reviews.csv'
KAGGLE_MODELS    = PROJECT_ROOT / 'models'
KAGGLE_MODELS.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Train / Test split ────────────────────────────────────────────────────────
TEST_SIZE = 0.2

# ── DistilBERT hyper-parameters ───────────────────────────────────────────────
BERT_MODEL_NAME = "distilbert-base-uncased"
BERT_MAX_LEN    = 128
BERT_BATCH_SIZE = 16
BERT_EPOCHS     = 3
BERT_LR         = 2e-5

# ── TF-IDF hyper-parameters ───────────────────────────────────────────────────
TFIDF_MAX_FEATURES = 5000

# ── Word2Vec hyper-parameters ─────────────────────────────────────────────────
W2V_VECTOR_SIZE = 100
W2V_WINDOW      = 5
W2V_MIN_COUNT   = 1
W2V_WORKERS     = 4

# ── Logistic Regression ───────────────────────────────────────────────────────
LR_MAX_ITER = 1000

# ── Insights: sentiment keyword lists ────────────────────────────────────────
INSIGHT_KEYWORDS = {
    "negative": ["slow", "bad", "terrible", "refund", "broken", "worst",
                 "expensive", "late", "poor", "awful", "disappointed",
                 "damaged", "missing", "lost", "cancel", "delay"],
    "positive": ["great", "fast", "love", "excellent", "amazing", "best",
                 "good", "cheap", "helpful", "easy", "perfect", "happy",
                 "satisfied", "recommend"],
}


In [4]:
# -------------
# preprocessing
# -------------

import re
import string
import pandas as pd

# Optional NLP tools
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are available
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))


# -----------------------------
# Text Cleaning Functions
# -----------------------------

def clean_text(text: str) -> str:
    """
    Basic text cleaning:
    - Lowercasing
    - Removing punctuation
    - Removing numbers
    - Removing extra spaces
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_stopwords(text: str) -> str:
    """
    Remove English stopwords
    """
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in STOPWORDS]
    return " ".join(filtered_tokens)


def preprocess_text(text: str) -> str:
    """
    Full preprocessing pipeline for a single text
    """
    text = clean_text(text)
    text = remove_stopwords(text)
    return text


# -----------------------------
# Dataset-Level Processing
# -----------------------------

def preprocess_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """
    Apply full preprocessing pipeline to dataset.

    Expected columns:
    - 'Review Title'
    - 'Review Text'
    - 'Rating'

    Output:
    - Adds 'Full Review'
    - Adds 'Sentiment'
    - Adds 'Cleaned Review'
    """

    # Drop unnecessary columns safely
    columns_to_drop = [
        'Reviewer Name',
        'Profile Link',
        'Review Count',
        'Review Date',
        'Date of Experience'
    ]
    data = data.drop(columns=[col for col in columns_to_drop if col in data.columns], errors='ignore')

    # Handle missing values
    data = data.dropna(subset=['Review Text', 'Rating'])
    data['Review Title'] = data['Review Title'].fillna('')
    data['Country'] = data['Country'].fillna('Unknown')

    # Extract numeric rating
    data['Rating'] = data['Rating'].astype(str).str.extract(r'(\d)').astype(float)
    data = data.dropna(subset=['Rating'])
    data['Rating'] = data['Rating'].astype(int)

    # Convert rating to sentiment
    def rating_to_sentiment(rating):
        if rating <= 2:
            return 0  # Negative
        elif rating >= 4:
            return 1  # Positive
        else:
            return None  # Neutral (drop later)

    data['Sentiment'] = data['Rating'].apply(rating_to_sentiment)
    data = data.dropna(subset=['Sentiment'])
    data['Sentiment'] = data['Sentiment'].astype(int)

    # Combine title + text
    data['Full Review'] = data['Review Title'] + " " + data['Review Text']

    # Apply text preprocessing
    data['Cleaned Review'] = data['Full Review'].apply(preprocess_text)

    # Optional: Review length (useful for analysis)
    data['Review Length'] = data['Cleaned Review'].apply(lambda x: len(x.split()))

    return data


## Preprocessing
---

In [5]:
import pandas as pd

print('Loading raw data...')
df_raw = pd.read_csv(KAGGLE_RAW_CSV, engine='python')
print(f'Raw shape: {df_raw.shape}')

print('Preprocessing...')
df_clean = preprocess_dataset(df_raw)
print(f'Clean shape: {df_clean.shape}')
df_clean.head()

Loading raw data...
Raw shape: (21214, 9)
Preprocessing...
Clean shape: (20170, 8)


,Country,Rating,Review Title,Review Text,Sentiment,Full Review,Cleaned Review,Review Length
0,US,1,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...",0,A Store That Doesn't Want to Sell Anything I r...,store doesnt want sell anything registered web...,57
1,GB,1,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,0,Had multiple orders one turned up and… Had mul...,multiple orders one turned and… multiple order...,36
2,GB,1,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,0,I informed these reprobates I informed these r...,informed reprobates informed reprobates would ...,55
3,AU,1,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,0,Advertise one price then increase it on websit...,advertise one price increase website bought am...,45
4,GB,1,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,0,If I could give a lower rate I would If I coul...,could give lower rate would could give lower r...,53


In [6]:
# Sentiment distribution
print(df_clean['Sentiment'].value_counts())
print(df_clean['Sentiment'].value_counts(normalize=True).mul(100).round(1))

Sentiment
0    14350
1     5820
Name: count, dtype: int64
Sentiment
0    71.1
1    28.9
Name: proportion, dtype: float64


In [7]:
KAGGLE_CLEAN_CSV.parent.mkdir(parents=True, exist_ok=True)

# Keep only the columns needed downstream
df_out = df_clean[['Cleaned Review', 'Sentiment']].rename(
    columns={'Cleaned Review': 'text', 'Sentiment': 'label'}
)
df_out.to_csv(KAGGLE_CLEAN_CSV, index=False)
print(f'Saved {len(df_out):,} rows → {KAGGLE_CLEAN_CSV}')

Saved 20,170 rows → /kaggle/working/data/processed/clean_reviews.csv


## Training
---

## 0. Shared Setup: Single Train/Test Split

In [9]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Fix all random seeds for reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

df = pd.read_csv(KAGGLE_CLEAN_CSV)
print(f'Loaded {len(df):,} reviews')

# ── THE SINGLE SPLIT ─────────────────────────────────────────────────────
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Save test split so app.py can load it for batch evaluation
test_df.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'test_split.csv', index=False)

X_train, y_train = train_df['text'].tolist(), train_df['label'].tolist()
X_test,  y_test  = test_df['text'].tolist(),  test_df['label'].tolist()

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print('Label distribution (train):', pd.Series(y_train).value_counts().to_dict())

Loaded 20,170 reviews
Train: 16,136 | Test: 4,034
Label distribution (train): {0: 11480, 1: 4656}


## 1. Logistic Regression + TF-IDF

In [11]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

tfidf_vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES)
X_train_tfidf    = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf     = tfidf_vectorizer.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=LR_MAX_ITER, random_state=SEED)
lr_tfidf.fit(X_train_tfidf, y_train)

preds_tfidf = lr_tfidf.predict(X_test_tfidf)
print('TF-IDF Accuracy:', accuracy_score(y_test, preds_tfidf))
print(classification_report(y_test, preds_tfidf, target_names=['Negative', 'Positive']))

with open(KAGGLE_MODELS / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
with open(KAGGLE_MODELS / 'lr_tfidf.pkl', 'wb') as f:
    pickle.dump(lr_tfidf, f)
print('TF-IDF artefacts saved ✓')

TF-IDF Accuracy: 0.9464551313832424
              precision    recall  f1-score   support

    Negative       0.95      0.98      0.96      2870
    Positive       0.95      0.86      0.90      1164

    accuracy                           0.95      4034
   macro avg       0.95      0.92      0.93      4034
weighted avg       0.95      0.95      0.95      4034

TF-IDF artefacts saved ✓


## 2. Logistic Regression + Word2Vec

In [13]:
import gensim

sentences_train = [t.split() for t in X_train]

w2v_model = gensim.models.Word2Vec(
    sentences_train,
    vector_size = W2V_VECTOR_SIZE,
    window      = W2V_WINDOW,
    min_count   = W2V_MIN_COUNT,
    workers     = W2V_WORKERS,
    seed        = SEED,
)

def get_sentence_vector(words, model):
    valid = [w for w in words if w in model.wv]
    return np.mean(model.wv[valid], axis=0) if valid else np.zeros(model.vector_size)

X_train_w2v = np.array([get_sentence_vector(t.split(), w2v_model) for t in X_train])
X_test_w2v  = np.array([get_sentence_vector(t.split(), w2v_model) for t in X_test])

lr_w2v = LogisticRegression(max_iter=LR_MAX_ITER, random_state=SEED)
lr_w2v.fit(X_train_w2v, y_train)

preds_w2v = lr_w2v.predict(X_test_w2v)
print('Word2Vec Accuracy:', accuracy_score(y_test, preds_w2v))
print(classification_report(y_test, preds_w2v, target_names=['Negative', 'Positive']))

w2v_model.save(str(KAGGLE_MODELS / 'w2v_model.gensim'))
with open(KAGGLE_MODELS / 'lr_w2v.pkl', 'wb') as f:
    pickle.dump(lr_w2v, f)
print('Word2Vec artefacts saved ✓')

Word2Vec Accuracy: 0.921417947446703
              precision    recall  f1-score   support

    Negative       0.93      0.97      0.95      2870
    Positive       0.90      0.81      0.86      1164

    accuracy                           0.92      4034
   macro avg       0.92      0.89      0.90      4034
weighted avg       0.92      0.92      0.92      4034

Word2Vec artefacts saved ✓


## 3. DistilBERT Fine-tuning

In [15]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.utils.class_weight import compute_class_weight

print(f'Device: {DEVICE}')

Device: cuda


In [16]:
# Dataset + DataLoader
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=BERT_MAX_LEN, padding='max_length',
            truncation=True, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

tokenizer    = DistilBertTokenizerFast.from_pretrained(BERT_MODEL_NAME)
train_loader = DataLoader(ReviewDataset(X_train, y_train, tokenizer),
                          batch_size=BERT_BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(ReviewDataset(X_test,  y_test,  tokenizer),
                          batch_size=BERT_BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 1009 | Test batches: 253


In [17]:
# Model, optimizer, scheduler, loss
bert_model = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=2
).to(DEVICE)

optimizer     = AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps   = len(train_loader) * BERT_EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)
class_weights = torch.tensor(
    compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train),
    dtype=torch.float,
).to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
# Training & evaluation helpers
def train_epoch(model, loader, optimizer, scheduler, loss_fn):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct    += (torch.argmax(logits, 1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = loss_fn(logits, labels)
            total_loss += loss.item()
            preds   = torch.argmax(logits, 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

In [19]:
# Main training loop
best_acc = 0.0

for epoch in range(1, BERT_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{BERT_EPOCHS}')
    train_loss, train_acc = train_epoch(bert_model, train_loader, optimizer, scheduler, loss_fn)
    test_loss,  test_acc, preds, labels = evaluate(bert_model, test_loader, loss_fn)
    print(f'  Train loss: {train_loss:.4f}  acc: {train_acc:.4f}')
    print(f'  Test  loss: {test_loss:.4f}  acc: {test_acc:.4f}')

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(bert_model.state_dict(), KAGGLE_MODELS / 'bert_best.pt')
        print(f'  ✓ Checkpoint saved (acc={best_acc:.4f})')

print(f'\nTraining complete. Best test accuracy: {best_acc:.4f}')


Epoch 1/3
  Train loss: 0.2575  acc: 0.9116
  Test  loss: 0.1756  acc: 0.9415
  ✓ Checkpoint saved (acc=0.9415)

Epoch 2/3
  Train loss: 0.1160  acc: 0.9732
  Test  loss: 0.1674  acc: 0.9581
  ✓ Checkpoint saved (acc=0.9581)

Epoch 3/3
  Train loss: 0.0644  acc: 0.9869
  Test  loss: 0.2009  acc: 0.9618
  ✓ Checkpoint saved (acc=0.9618)

Training complete. Best test accuracy: 0.9618


In [20]:
# Final evaluation with the best checkpoint
from sklearn.metrics import classification_report

bert_model.load_state_dict(torch.load(KAGGLE_MODELS / 'bert_best.pt', map_location=DEVICE))
_, final_acc, final_preds, final_labels = evaluate(bert_model, test_loader, loss_fn)

print(f'BERT Final Accuracy: {final_acc:.4f}')
print(classification_report(final_labels, final_preds, target_names=['Negative', 'Positive']))

BERT Final Accuracy: 0.9618
              precision    recall  f1-score   support

    Negative       0.98      0.97      0.97      2870
    Positive       0.93      0.94      0.93      1164

    accuracy                           0.96      4034
   macro avg       0.95      0.96      0.95      4034
weighted avg       0.96      0.96      0.96      4034

